# 8.02 SqliteSaver · AsyncSqliteSaver — 파일 기반 체크포인터

`langgraph-checkpoint-sqlite` 패키지가 제공하는 동기 `SqliteSaver` 와 비동기 `AsyncSqliteSaver`.
**단일 파일 DB** 이므로 서비스 기동이 필요 없고, 노트북 · 로컬 에이전트 · 엣지 배포에 잘 맞는다.

## 학습 목표

- `SqliteSaver.from_conn_string(":memory:")` / `"checkpoints.sqlite"` 로 인스턴스 만들기
- `with` 블록 밖에서 체크포인터를 오래 쓰고 싶을 때 `sqlite3.connect()` + `SqliteSaver(conn)` 직접 구성
- thread_id 기반 복원이 **프로세스를 다시 띄워도** 유지됨을 확인
- `AsyncSqliteSaver` 를 `await` 워크플로우에 붙이기

## 언제 쓰나

- 단일 프로세스 로컬 에이전트 — 서비스 없이 바로 영속 원할 때
- 데모 / PoC — 재현 가능한 대화 로그가 파일 하나로 보관됨
- 엣지(edge) · 데스크톱 앱 — 네트워크 DB가 부담스러운 환경

⚠️ SQLite 는 **동시 쓰기** 에 약하다. 프로세스가 여럿이거나 트래픽이 많다면 Postgres 로 넘어간다.


## 8.02.1 환경 설정 · 서비스 연결

`langgraph-checkpoint-sqlite` 설치 후, 실제 파일 경로에 연결을 한 번 열어서 probe 한다.
`.local/` 아래 임시 파일을 쓰므로 저장소에 흔적이 남지 않는다.

In [ ]:
# !pip install -U langgraph langgraph-checkpoint-sqlite langchain langchain-openai python-dotenv

import os, sqlite3, pathlib
from dotenv import load_dotenv
load_dotenv(override=True)

assert os.environ.get("OPENAI_API_KEY"), "OPENAI_API_KEY 필요"

# 로컬 파일 경로 probe — 디렉토리 생성 + 실제 connect/close
DB_DIR = pathlib.Path("/Users/baem1n/Workspace/langchain-langgraph-deepagents-notebooks/.local")
DB_DIR.mkdir(parents=True, exist_ok=True)
DB_PATH = str(DB_DIR / "ckpt_sqlite_demo.sqlite")

# 접근 권한 · 디스크 확인 — 실패 시 OperationalError
conn0 = sqlite3.connect(DB_PATH)
conn0.execute("SELECT 1").fetchone()
conn0.close()

from langgraph.checkpoint.sqlite import SqliteSaver
from langgraph.checkpoint.sqlite.aio import AsyncSqliteSaver
print("sqlite path OK:", DB_PATH)
print("classes:", SqliteSaver.__name__, "/", AsyncSqliteSaver.__name__)


## 8.02.2 기본 사용법 — 파일 기반 영속 체크포인트

`SqliteSaver(conn)` 에 직접 `sqlite3.Connection` 을 넘기면 `with` 블록 밖에서도 오래 살아있는 saver 가 된다.
`check_same_thread=False` 는 **그래프를 만든 스레드 외부에서도** 같은 커넥션을 쓸 수 있게 한다 (Jupyter / 서버에서 필수).

In [ ]:
import operator
from typing import Annotated
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages

class ChatState(TypedDict):
    messages: Annotated[list, add_messages]
    turn: Annotated[int, operator.add]   # 호출마다 +1 누적 (체크포인터 복원 확인용)

def respond(state: ChatState) -> ChatState:
    last = state["messages"][-1].content if state["messages"] else "(empty)"
    next_turn = state.get("turn", 0) + 1
    return {
        "messages": [{"role": "assistant", "content": f"echo[{next_turn}]: {last}"}],
        "turn": 1,
    }

builder = StateGraph(ChatState)
builder.add_node("respond", respond)
builder.add_edge(START, "respond")
builder.add_edge("respond", END)

conn = sqlite3.connect(DB_PATH, check_same_thread=False)
saver = SqliteSaver(conn)  # 최초 호출 시 필요한 테이블이 lazily 생성됨
graph = builder.compile(checkpointer=saver)
print("compiled:", graph.checkpointer.__class__.__name__)


## 8.02.3 thread_id 스코프와 복원

SQLite 체크포인터는 **프로세스 재시작 후에도** 같은 파일을 열면 이전 대화가 그대로 살아난다.
아래에서는 같은 파일에 `SqliteSaver` 를 **다시 열어서** 이전 thread 의 상태가 보이는 것을 확인한다.

In [ ]:
cfg = {"configurable": {"thread_id": "sqlite-demo-1"}}

for msg in ["첫 메시지", "두 번째", "세 번째"]:
    out = graph.invoke({"messages": [{"role": "user", "content": msg}]}, cfg)
print("현재 turn =", out["turn"])

# 새 커넥션 + 새 saver 로 같은 파일을 다시 연다 (프로세스 재시작과 유사)
conn2 = sqlite3.connect(DB_PATH, check_same_thread=False)
saver2 = SqliteSaver(conn2)
graph2 = builder.compile(checkpointer=saver2)

snap = graph2.get_state(cfg)
print("재오픈 후 turn =", snap.values.get("turn"))
print("messages 수 =", len(snap.values.get("messages", [])))


## 8.02.4 `get_state_history` · time travel

SQLite 도 InMemory 와 동일한 API 를 제공한다. 과거 체크포인트를 찾아 그 시점으로 분기할 수 있다.

In [ ]:
history = list(graph.get_state_history(cfg))
print(f"체크포인트 수: {len(history)}")
for i, h in enumerate(history[:5]):
    print(f"  [{i}] turn={h.values.get('turn')} next={h.next}")

# turn==1 (첫 응답 직후) 로 되돌아가 분기
target = next(h for h in reversed(history) if h.values.get("turn") == 1 and h.next == ())
branched = graph.invoke(
    {"messages": [{"role": "user", "content": "여기서 다른 길로"}]},
    target.config,
)
print("\nbranched turn =", branched["turn"])
# fork 후 get_state 는 새로 만든 branch 체크포인트를 반환. 원본 히스토리는 트리에 보존된다.
print("latest(branch 후) turn =", graph.get_state(cfg).values["turn"])
print("전체 history 길이(원본+branch) =", len(list(graph.get_state_history(cfg))))


## 8.02.5 `AsyncSqliteSaver` — 비동기 워크플로우

asyncio 앱(예: FastAPI, Telegram bot, Discord bot)에서는 async saver 를 써야 한다. `from_conn_string` 이 async context manager 를 돌려주므로 `async with` 로 수명 관리.

In [ ]:
import asyncio

async def run_async_demo():
    async with AsyncSqliteSaver.from_conn_string(DB_PATH) as asaver:
        agraph = builder.compile(checkpointer=asaver)
        cfg_a = {"configurable": {"thread_id": "async-sqlite-1"}}
        for msg in ["async 1", "async 2"]:
            out = await agraph.ainvoke(
                {"messages": [{"role": "user", "content": msg}]},
                cfg_a,
            )
        snap = await agraph.aget_state(cfg_a)
        print("async turn =", snap.values["turn"])

        history = []
        async for h in agraph.aget_state_history(cfg_a):
            history.append(h)
        print("async history 길이 =", len(history))

await run_async_demo()


## 8.02.6 `create_agent` 에서 파일 기반 메모리

LangChain 에이전트에 파일 영속 메모리를 붙이는 가장 짧은 경로.

In [ ]:
from langchain.agents import create_agent
from langchain_openai import ChatOpenAI

conn_agent = sqlite3.connect(str(DB_DIR / "ckpt_agent.sqlite"), check_same_thread=False)
agent = create_agent(
    model=ChatOpenAI(model="gpt-4.1-mini", temperature=0),
    tools=[],
    checkpointer=SqliteSaver(conn_agent),
    system_prompt="너는 짧게 답하는 메모리 에이전트야.",
)

cfg = {"configurable": {"thread_id": "agent-sqlite-1"}}
r1 = agent.invoke({"messages": [{"role": "user", "content": "내 취미는 클라이밍이야"}]}, cfg)
print("1:", r1["messages"][-1].content[:120])

r2 = agent.invoke({"messages": [{"role": "user", "content": "내 취미가 뭐였지?"}]}, cfg)
print("2:", r2["messages"][-1].content[:120])


## 체크리스트

- [ ] `sqlite3.connect(..., check_same_thread=False)` — Jupyter/서버에서 필수
- [ ] 파일 경로는 **쓰기 가능** · 충분한 디스크 공간 확보
- [ ] 동시 쓰기가 잦다면 Postgres 로 — SQLite 는 writer 1 개가 적절
- [ ] 비동기 경로는 반드시 `AsyncSqliteSaver` (동기 saver 를 await 하지 말 것)
- [ ] 파일을 옮기면 thread_id 도 함께 이동 — 별도 export 불필요

## 다음

- `03_postgres.ipynb` — 멀티 writer 지원, `.setup()` 으로 스키마 생성
- `04_cosmosdb.ipynb` — Azure NoSQL 백엔드
- `09_stores/` — 스레드 간 공유 지식 (`BaseStore`)
- `docs/skills/langgraph-persistence.md`

## 참고

- `langgraph-checkpoint-sqlite`: https://pypi.org/project/langgraph-checkpoint-sqlite/
- LangGraph 체크포인터 개요: https://langchain-ai.github.io/langgraph/how-tos/persistence/
